In [9]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

/var/folders/y9/yzw2lv891kn9kfdx6qhrb39w0000gn/T/ipykernel_78563/3777615979.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython.display
  from IPython.core.display import display, HTML


# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [10]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [11]:
## Read Data for the Fraudulent Email Kaggle Challenge
data_train = pd.read_csv("/Users/laxmigupte/Desktop/Ironhack/week7/D1/lab-natural-language-processing/data/kg_train.csv",encoding='latin-1')

# Reduce the training set to speed up development. 
# Modify for final system
data_train = data_train.head(1000)
print(data_train.shape)
data_train.fillna("",inplace=True)

(1000, 2)


### Let's divide the training and test set into two partitions

In [12]:
# Your code
from sklearn.model_selection import train_test_split

# Features and labels
X = data_train["text"]
y = data_train["label"]

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)


(800,)
(200,)


## Data Preprocessing

In [13]:
import string
from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [14]:
# 0. Import required libraries
import re

- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [15]:
# 1. Remove inline JavaScript/CSS
def remove_script(text):

    pattern = r'<script.*?>.*?</script>'

    cleaned = re.sub(pattern, '', text, flags=re.DOTALL)

    return cleaned

In [16]:
# 2. Remove HTML comments
def remove_comments(text):

    pattern = r'<!--.*?-->'

    cleaned = re.sub(pattern, '', text, flags=re.DOTALL)

    return cleaned

In [17]:
# 3. Remove HTML tags
def remove_tags(text):

    pattern = r'<.*?>'

    cleaned = re.sub(pattern, '', text)

    return cleaned

In [18]:
# 4. Remove special characters
def remove_special(text):

    cleaned = re.sub(r'[^a-zA-Z0-9\s]', '', text)

    return cleaned

In [19]:
# 5. Remove numbers
def remove_numbers(text):

    cleaned = re.sub(r'\d+', '', text)

    return cleaned

In [20]:
# 6. Remove single characters
def remove_single_char(text):

    cleaned = re.sub(r'\s+[a-zA-Z]\s+', ' ', text)

    return cleaned

In [21]:
# 7.Remove single characters from start
def remove_single_start(text):

    cleaned = re.sub(r'\^[a-zA-Z]\s+', '', text)

    return cleaned

In [22]:
# 8. Replace multiple spaces
def remove_multiple_spaces(text):

    cleaned = re.sub(r'\s+', ' ', text)

    return cleaned

In [23]:
# 9. Remove prefixed b
def remove_prefixed_b(text):

    cleaned = re.sub(r'^b\s+', '', text)

    return cleaned

In [24]:
# 10. Convert to lowercase
def lower_case(text):

    return text.lower()

In [25]:
# 11. Create ONE preprocessing function
def preprocess(text):

    text = remove_script(text)

    text = remove_comments(text)

    text = remove_tags(text)

    text = remove_special(text)

    text = remove_numbers(text)

    text = remove_single_char(text)

    text = remove_single_start(text)

    text = remove_multiple_spaces(text)

    text = remove_prefixed_b(text)

    text = lower_case(text)

    return text


In [26]:
# 12. Test it
sample = """
<html>

<script>
alert('hello')
</script>

<!-- comment -->

<h1>Hello World 123!!!</h1>

</html>
"""

print(preprocess(sample))

 hello world 


## Now let's work on removing stopwords
Remove the stopwords.

In [27]:
# Your code
from nltk.corpus import stopwords

In [28]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):

    words = text.split()

    filtered_words = [
        word for word in words
        if word not in stop_words
    ]

    return " ".join(filtered_words)

In [29]:
# Test
sample = "this is a very good movie"

print(remove_stopwords(sample))

good movie


## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [30]:
# Your code
from nltk.stem import WordNetLemmatizer

In [31]:
# Create lemmatizer object
lemmatizer = WordNetLemmatizer()

In [32]:
# Create function
def lemmatize_text(text):

    words = text.split()

    lemmatized_words = [
        lemmatizer.lemmatize(word)
        for word in words
    ]

    return " ".join(lemmatized_words)

In [33]:
# Test
sample = "running cats studies"

print(lemmatize_text(sample))

running cat study


## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [34]:
# Your code
from sklearn.feature_extraction.text import CountVectorizer

In [35]:
# Create sample corpus
corpus = [
    "I love machine learning",
    "Machine learning is fun",
    "I love NLP"
]

In [36]:
# Create vectorizer
vectorizer = CountVectorizer()

In [37]:
# Fit and transform
X = vectorizer.fit_transform(corpus)

In [38]:
# Show vocabulary
print(vectorizer.get_feature_names_out())

['fun' 'is' 'learning' 'love' 'machine' 'nlp']


In [39]:
# Show vectors
print(X.toarray())

[[0 0 1 1 1 0]
 [1 1 1 0 1 0]
 [0 0 0 1 0 1]]


In [45]:
# Clean and preprocessed data
data_train['preprocessed_text'] = data_train['text'].apply(preprocess)

## Extra features

In [47]:
# We add to the original dataframe two additional indicators

money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])

suspicious_words = "|".join([
    "free",
    "cheap",
    "sex",
    "money",
    "account",
    "bank",
    "fund",
    "transfer",
    "transaction",
    "win",
    "deposit",
    "password"
])

data_train['money_mark'] = data_train['preprocessed_text'].str.contains(
    money_simbol_list
)*1

data_train['suspicious_words'] = data_train['preprocessed_text'].str.contains(
    suspicious_words
)*1

data_train['text_len'] = data_train['preprocessed_text'].apply(
    lambda x: len(x)
)

data_train.head()

,text,label,preprocessed_text,money_mark,suspicious_words,text_len
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1,dear sir strictly private business proposal am...,1,1,2196
1,Will do.,0,will do,0,0,7
2,Nora--Cheryl has emailed dozens of memos about...,0,noracheryl has emailed dozens of memos about h...,0,0,192
3,Dear Sir=2FMadam=2C I know that this proposal ...,1,dear sirfmadamc know that this proposal might ...,1,1,2018
4,fyi,0,fyi,0,0,3


## How would work the Bag of Words with Count Vectorizer concept?

In [48]:
# Your code
from sklearn.feature_extraction.text import CountVectorizer

# Create vectorizer
cv = CountVectorizer(max_features=5000)

# Convert text into vectors
X_bow = cv.fit_transform(data_train['preprocessed_text'])

# Print matrix shape
print(X_bow.shape)

(1000, 5000)


In [49]:
#See vocabulary
print(cv.get_feature_names_out()[:20])

['aae' 'abacha' 'abandoned' 'abbas' 'abdull' 'abedin' 'abidjan'
 'abidjancote' 'abilateral' 'ability' 'able' 'abm' 'abn' 'aboard'
 'abortive' 'about' 'aboute' 'aboutthe' 'above' 'abroad']


In [50]:
#See vectors
print(X_bow.toarray()[0])

[0 0 1 ... 0 0 0]


## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [51]:
# Your code
from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF vectorizer
tfidf = TfidfVectorizer(max_features=5000)

# Convert text into TF-IDF vectors
X_tfidf = tfidf.fit_transform(data_train['preprocessed_text'])

# Print shape
print(X_tfidf.shape)

(1000, 5000)


## And the Train a Classifier?

In [52]:
# Your code
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# Features
X = X_tfidf

# Labels
y = data_train['label']

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Create model
model = MultinomialNB()

# Train model
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Accuracy
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.945


### Extra Task - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

The classifier can not be changed!!! It must be the MultinimialNB with default parameters!

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [53]:
# Your code
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
from scipy.sparse import hstack

# Labels
y = data_train['label']

# Extra numerical features
extra_features = data_train[['money_mark', 'suspicious_words', 'text_len']].values

# ---------------------------------------------------
# 1. BAG OF WORDS ONLY
# ---------------------------------------------------

cv = CountVectorizer(max_features=5000)

X_bow = cv.fit_transform(data_train['preprocessed_text'])

X_train, X_test, y_train, y_test = train_test_split(
    X_bow,
    y,
    test_size=0.2,
    random_state=42
)

model = MultinomialNB()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("BOW Accuracy:", accuracy_score(y_test, y_pred))

# ---------------------------------------------------
# 2. TF-IDF ONLY
# ---------------------------------------------------

tfidf = TfidfVectorizer(max_features=5000)

X_tfidf = tfidf.fit_transform(data_train['preprocessed_text'])

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=42
)

model = MultinomialNB()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("TF-IDF Accuracy:", accuracy_score(y_test, y_pred))

# ---------------------------------------------------
# 3. BOW + EXTRA FEATURES
# ---------------------------------------------------

X_bow_extra = hstack([X_bow, extra_features])

X_train, X_test, y_train, y_test = train_test_split(
    X_bow_extra,
    y,
    test_size=0.2,
    random_state=42
)

model = MultinomialNB()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("BOW + Extra Features Accuracy:", accuracy_score(y_test, y_pred))

# ---------------------------------------------------
# 4. TF-IDF + EXTRA FEATURES
# ---------------------------------------------------

X_tfidf_extra = hstack([X_tfidf, extra_features])

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf_extra,
    y,
    test_size=0.2,
    random_state=42
)

model = MultinomialNB()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("TF-IDF + Extra Features Accuracy:", accuracy_score(y_test, y_pred))

BOW Accuracy: 0.955
TF-IDF Accuracy: 0.945
BOW + Extra Features Accuracy: 0.955
TF-IDF + Extra Features Accuracy: 0.91


BOW combined with engineered features achieved the best performance.